In [6]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd

from src.etl.loader import load_all_datasets

datasets = load_all_datasets()

print(f"Loaded {len(datasets)} datasets.")

RAW PATH: C:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\raw
SUPPORTING PATH: C:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\raw\supporting datasets
Loaded 12 datasets.


In [7]:
for name, df in datasets.items():
    print("=" * 80)
    print(name.upper())
    print("=" * 80)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\n")

COMPANIES
Shape: (92, 12)

Columns:
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']

Data Types:
id                  object
company_logo        object
company_name        object
chart_link          object
about_company       object
website             object
nse_profile         object
bse_profile         object
face_value         float64
book_value         float64
roce_percentage    float64
roe_percentage     float64
dtype: object


PROFITANDLOSS
Shape: (1276, 15)

Columns:
['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']

Data Types:
id                     int64
company_id            object
year                  object
sales                  int64
expenses               int64
operating_profit     flo

In [8]:
for name, df in datasets.items():

    print("=" * 90)
    print(name.upper())
    print("=" * 90)

    print(f"Rows    : {len(df)}")
    print(f"Columns : {len(df.columns)}")

    print("\nNull Values")
    print(df.isnull().sum())

    print("\nDuplicate Rows:", df.duplicated().sum())

    if "id" in df.columns:
        print("Duplicate IDs :", df["id"].duplicated().sum())

    print("\n")

COMPANIES
Rows    : 92
Columns : 12

Null Values
id                 0
company_logo       1
company_name       0
chart_link         0
about_company      0
website            1
nse_profile        1
bse_profile        1
face_value         1
book_value         1
roce_percentage    1
roe_percentage     2
dtype: int64

Duplicate Rows: 0
Duplicate IDs : 0


PROFITANDLOSS
Rows    : 1276
Columns : 15

Null Values
id                     0
company_id             0
year                   0
sales                  0
expenses               0
operating_profit      13
opm_percentage        15
other_income           0
interest               0
depreciation           0
profit_before_tax      0
tax_percentage        95
net_profit             0
eps                    5
dividend_payout      103
dtype: int64

Duplicate Rows: 0
Duplicate IDs : 0


BALANCESHEET
Rows    : 1312
Columns : 13

Null Values
id                   0
company_id           0
year                 0
equity_capital       0
reserves           

In [9]:
db_path = PROJECT_ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

print("Database created successfully.")

Database created successfully.


In [10]:
print(db_path)
print(db_path.exists())

c:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\db\nifty100.db
True


In [11]:
schema_path = PROJECT_ROOT / "data" / "db" / "schema.sql"

with open(schema_path, "r", encoding="utf-8") as file:
    cursor.executescript(file.read())

conn.commit()

print("Schema executed successfully.")

Schema executed successfully.


In [12]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn)

tables

,name
0,analysis
1,balancesheet
2,cashflow
3,companies
4,documents
5,financial_ratios
6,market_cap
7,peer_groups
8,profitandloss
9,prosandcons


In [20]:
cursor.execute("DELETE FROM profitandloss;")
cursor.execute("DELETE FROM companies;")
conn.commit()

In [21]:
companies = datasets["companies"]

companies.to_sql(
    "companies",
    conn,
    if_exists="append",
    index=False
)

print(f"Loaded {len(companies)} rows into companies.")

Loaded 92 rows into companies.


In [22]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM companies;",
    conn
)

,total_rows
0,92


In [23]:
profitandloss = datasets["profitandloss"]

profitandloss.to_sql(
    "profitandloss",
    conn,
    if_exists="append",
    index=False
)

print(f"Loaded {len(profitandloss)} rows into profitandloss.")

Loaded 1177 rows into profitandloss.


In [16]:
companies_db = pd.read_sql(
    "SELECT id FROM companies",
    conn
)

print(companies_db.shape)
companies_db.head()

(92, 1)


,id
0,ABB
1,ADANIENSOL
2,ADANIENT
3,ADANIGREEN
4,ADANIPORTS


In [17]:
db_ids = set(companies_db["id"])
pl_ids = set(datasets["profitandloss"]["company_id"])

missing = sorted(pl_ids - db_ids)

print("Missing IDs:", missing)
print("Count:", len(missing))

Missing IDs: ['ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VBL', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']
Count: 8


In [18]:
print(repr(companies_db.iloc[0]["id"]))
print(repr(datasets["profitandloss"].iloc[0]["company_id"]))

'ABB'
'ABB'


In [19]:
valid_ids = set(datasets["companies"]["id"])

print("Removing orphan records...\n")

for table, df in datasets.items():

    if "company_id" in df.columns:

        before = len(df)

        datasets[table] = df[df["company_id"].isin(valid_ids)].copy()

        after = len(datasets[table])

        print(f"{table:<20} Removed: {before-after:>3} rows | Remaining: {after}")

Removing orphan records...

profitandloss        Removed:  99 rows | Remaining: 1177
balancesheet         Removed:  85 rows | Remaining: 1227
cashflow             Removed:  96 rows | Remaining: 1091
analysis             Removed:   4 rows | Remaining: 16
documents            Removed: 128 rows | Remaining: 1457
prosandcons          Removed:   2 rows | Remaining: 14
financial_ratios     Removed:  24 rows | Remaining: 1160
market_cap           Removed:   0 rows | Remaining: 552
peer_groups          Removed:   0 rows | Remaining: 56
sectors              Removed:   0 rows | Remaining: 92
stock_prices         Removed:   0 rows | Remaining: 5520


In [28]:
tables = [
    "companies",
    "profitandloss",
    "balancesheet",
    "cashflow",
    "analysis",
    "documents",
    "prosandcons",
    "financial_ratios",
    "market_cap",
    "peer_groups",
    "sectors",
    "stock_prices"
]

for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS rows FROM {table}",
        conn
    ).iloc[0, 0]

    print(f"{table:<20} {count}")

companies            92
profitandloss        1177
balancesheet         1227
cashflow             1091
analysis             16
documents            1457
prosandcons          14
financial_ratios     1160
market_cap           552
peer_groups          56
sectors              92
stock_prices         5520


In [30]:
load_audit = []

for table in [
    "companies",
    "profitandloss",
    "balancesheet",
    "cashflow",
    "analysis",
    "documents",
    "prosandcons",
    "financial_ratios",
    "market_cap",
    "peer_groups",
    "sectors",
    "stock_prices"
]:
    rows = pd.read_sql(
        f"SELECT COUNT(*) AS rows FROM {table}",
        conn
    ).iloc[0, 0]

    load_audit.append({
        "table": table,
        "rows_loaded": rows,
        "status": "Success"
    })

audit = pd.DataFrame(load_audit)
audit

,table,rows_loaded,status
0,companies,92,Success
1,profitandloss,1177,Success
2,balancesheet,1227,Success
3,cashflow,1091,Success
4,analysis,16,Success
5,documents,1457,Success
6,prosandcons,14,Success
7,financial_ratios,1160,Success
8,market_cap,552,Success
9,peer_groups,56,Success


In [31]:
audit.to_csv(
    PROJECT_ROOT / "output" / "load_audit.csv",
    index=False
)

print("Load audit saved successfully.")

Load audit saved successfully.
